In [1]:
print("ram ram")

ram ram


In [3]:
# =============================================================================
# FORCE INSTALL into the CURRENT KERNEL's Python
# =============================================================================
import sys
import subprocess

print("Current kernel Python:", sys.executable)
print("-" * 60)

packages = [
    "langgraph",
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-groq",
    "langchain-google-genai",
    "litellm",
    "pypdf",
    "pillow",
    "google-generativeai",
    "grandalf",
    "ipython"
]

for pkg in packages:
    print(f"Installing {pkg} ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", pkg
    ])

print("\n✅ Installation finished into THIS kernel.")
print(">>> Ab KERNEL RESTART karo (Kernel → Restart Kernel), phir Cell 2 se aage chalao.")

Current kernel Python: c:\Users\Administrator\Desktop\MAI (19-08-26)\New folder\backend\venv\Scripts\python.exe
------------------------------------------------------------
Installing langgraph ...
Installing langchain ...
Installing langchain-core ...
Installing langchain-community ...
Installing langchain-groq ...


CalledProcessError: Command '['c:\\Users\\Administrator\\Desktop\\MAI (19-08-26)\\New folder\\backend\\venv\\Scripts\\python.exe', '-m', 'pip', 'install', '--upgrade', '--force-reinstall', 'langchain-groq']' returned non-zero exit status 1.

In [ ]:
# =============================================================================
# Cell 2: LiteLLM / LLM Gateway Configuration
# =============================================================================

from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from typing import Annotated, TypedDict, List, Optional, Dict, Any
import base64
from pathlib import Path
from IPython.display import display, Image, Markdown
import io
import os

# Keys Setup
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

CONSTRUCTION_DB_URL = os.getenv("CONSTRUCTION_DB_URL")
MANUFACTURING_DB_URL = os.getenv("MANUFACTURING_DB_URL")

# ------------------------------------------------------------
# Fast text / routing model (Groq Llama-3)
# ------------------------------------------------------------
llm_chat = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    max_tokens=2048,
)

# ------------------------------------------------------------
# Multimodal / Vision model (Gemini)
# ------------------------------------------------------------
llm_vision = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",          # or "gemini-1.5-flash"
    temperature=0.1,
    max_output_tokens=2048,
)

print("✅ LLM clients initialized:")
print(f"   • Chat / Routing : {llm_chat.model_name}")
print(f"   • Vision         : {llm_vision.model}")

ModuleNotFoundError: No module named 'langchain_groq'

In [ ]:
# =============================================================================
# Cell 3: Tool Definitions (summary_tools)
# =============================================================================

from langchain_core.tools import tool
from pypdf import PdfReader
from PIL import Image as PILImage
import tempfile
import traceback

@tool
def summarize_text_document(file_path: str) -> str:
    """
    Extract text from a PDF or plain-text (.txt) file and produce a structured summary.
    
    Args:
        file_path: Absolute or relative path to the document (PDF or TXT).
    
    Returns:
        A concise, well-structured summary of the document content.
    """
    path = Path(file_path)
    if not path.exists():
        return f"Error: File not found → {file_path}"

    try:
        text = ""
        if path.suffix.lower() == ".pdf":
            reader = PdfReader(str(path))
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        elif path.suffix.lower() in {".txt", ".md", ".csv"}:
            text = path.read_text(encoding="utf-8", errors="ignore")
        else:
            return f"Unsupported file type: {path.suffix}. Supported: .pdf, .txt, .md, .csv"

        if not text.strip():
            return "The document appears to be empty or contains no extractable text."

        # Truncate extremely long documents to stay within context limits
        max_chars = 30_000
        if len(text) > max_chars:
            text = text[:max_chars] + "\n\n[... document truncated for summarization ...]"

        prompt = f"""You are an expert document analyst. 
Produce a clear, structured summary of the following document.
Include:
1. Main topic / purpose
2. Key points (bullet list)
3. Important data, figures or conclusions
4. Overall takeaway

Document content:
{text}
"""
        response = llm_chat.invoke([HumanMessage(content=prompt)])
        return response.content

    except Exception as e:
        return f"Error while processing document: {str(e)}\n{traceback.format_exc()}"


@tool
def analyze_chart_image(image_path: str) -> str:
    """
    Analyze a chart / graph image (PNG, JPG, JPEG, WEBP) using a multimodal vision model.
    Extracts key trends, data points, axes, and visual insights.
    
    Args:
        image_path: Path to the image file containing a chart or graph.
    
    Returns:
        Structured analysis of the chart (trends, key values, insights).
    """
    path = Path(image_path)
    if not path.exists():
        return f"Error: Image not found → {image_path}"

    if path.suffix.lower() not in {".png", ".jpg", ".jpeg", ".webp", ".gif"}:
        return f"Unsupported image type: {path.suffix}"

    try:
        # Load and optionally resize very large images
        img = PILImage.open(path).convert("RGB")
        max_dim = 2048
        if max(img.size) > max_dim:
            img.thumbnail((max_dim, max_dim), PILImage.Resampling.LANCZOS)

        # Convert to base64 for the multimodal message
        buffered = io.BytesIO()
        img.save(buffered, format="PNG")
        img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")

        # Multimodal message for Gemini
        message = HumanMessage(
            content=[
                {
                    "type": "text",
                    "text": (
                        "You are an expert data visualization analyst. "
                        "Carefully examine the provided chart/graph image and return a structured analysis containing:\n"
                        "1. Chart type (bar, line, pie, scatter, etc.)\n"
                        "2. Title / subject (if present)\n"
                        "3. Axes labels and units\n"
                        "4. Key data points and values visible\n"
                        "5. Main trends, patterns or anomalies\n"
                        "6. Concise insight / takeaway\n\n"
                        "Be precise and quantitative wherever possible."
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{img_b64}"},
                },
            ]
        )

        response = llm_vision.invoke([message])
        return response.content

    except Exception as e:
        return f"Error while analyzing image: {str(e)}\n{traceback.format_exc()}"


# Collect tools
summary_tools = [summarize_text_document, analyze_chart_image]

print("✅ Tools defined:")
for t in summary_tools:
    print(f"   • {t.name}")

In [ ]:
# =============================================================================
# Cell 4: Define LangGraph State & Node Logic
# =============================================================================

from typing_extensions import Annotated
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# ------------------------------------------------------------
# Agent State
# ------------------------------------------------------------
class AgentState(TypedDict):
    """State of the Summary Agent."""
    messages: Annotated[list, add_messages]


# ------------------------------------------------------------
# Agent node – binds tools and decides whether to call them
# ------------------------------------------------------------
# Bind tools to the chat model so it can emit tool calls
llm_with_tools = llm_chat.bind_tools(summary_tools)

SYSTEM_PROMPT = """You are a helpful Summary Agent with two specialized capabilities:

1. **Document Summarization** – When the user provides a path to a PDF or text file, 
   use the `summarize_text_document` tool to extract and summarize its content.

2. **Chart / Image Analysis** – When the user provides a path to a chart or graph image 
   (PNG/JPG), use the `analyze_chart_image` tool to extract trends, data points and insights.

For ordinary conversation (greetings, questions about your abilities, general chat) 
answer directly without calling tools.

Always be concise, professional and helpful. When a tool returns a result, 
present it clearly to the user and offer follow-up assistance.
"""

def summary_agent(state: AgentState) -> Dict[str, Any]:
    """
    Core agent node. Receives conversation history, decides whether to call a tool
    or reply directly.
    """
    messages = state["messages"]
    
    # Prepend system prompt on every turn (stateless models)
    full_messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    
    response = llm_with_tools.invoke(full_messages)
    return {"messages": [response]}


# Tool execution node
tool_node = ToolNode(summary_tools)

print("✅ Agent node and ToolNode created.")

In [ ]:
# =============================================================================
# Cell 5: Build & Compile the Graph
# =============================================================================

from langgraph.graph import StateGraph, START, END

# Create the graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("summary_agent", summary_agent)
workflow.add_node("summary_tools", tool_node)

# Entry point
workflow.add_edge(START, "summary_agent")

# Conditional routing: if the agent emitted a tool call → go to tools, else END
workflow.add_conditional_edges(
    "summary_agent",
    tools_condition,               # built-in helper that inspects AIMessage.tool_calls
    {
        "tools": "summary_tools",  # tool call detected
        END: END,                  # no tool call → finish
    },
)

# After tools finish, loop back to the agent so it can formulate the final answer
workflow.add_edge("summary_tools", "summary_agent")

# Compile with an in-memory checkpointer (enables multi-turn memory)
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)

print("✅ Graph compiled successfully with MemorySaver checkpointer.")

In [ ]:
# =============================================================================
# Cell 6: Graph Visualization
# =============================================================================

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Could not render Mermaid diagram (missing dependencies or display backend).")
    print("ASCII representation:")
    print(graph.get_graph().draw_ascii())

In [ ]:
# =============================================================================
# Cell 7: Testing & Execution Examples
# =============================================================================

from langchain_core.messages import HumanMessage
import uuid

def run_agent(user_input: str, thread_id: str = None, verbose: bool = True):
    """
    Helper to invoke the compiled graph with a single user message.
    Maintains conversation state via the checkpointer.
    """
    if thread_id is None:
        thread_id = str(uuid.uuid4())
    
    config = {"configurable": {"thread_id": thread_id}}
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"USER  → {user_input}")
        print(f"{'='*60}")
    
    # Stream events for better observability (optional)
    final_response = None
    for event in graph.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
        stream_mode="values",
    ):
        last_msg = event["messages"][-1]
        if isinstance(last_msg, AIMessage) and not last_msg.tool_calls:
            final_response = last_msg.content
    
    if verbose and final_response:
        print(f"AGENT → {final_response}\n")
    
    return final_response, thread_id


# ------------------------------------------------------------------
# Test Case 1: General Conversation
# ------------------------------------------------------------------
print("\n📌 TEST CASE 1 – General Conversation")
resp1, tid1 = run_agent("Hi! What can you do?")

# ------------------------------------------------------------------
# Test Case 2: Document Summarization
# ------------------------------------------------------------------
# Replace with a real PDF path on your machine / Colab upload
print("\n📌 TEST CASE 2 – Document Summarization")
sample_pdf = "/path/to/your/sample.pdf"   # <-- CHANGE THIS

# Uncomment the line below when you have a real file:
# resp2, tid2 = run_agent(f"Please summarize this document: {sample_pdf}", thread_id=tid1)

print("(Skipped – provide a real PDF path and uncomment the call)")

# ------------------------------------------------------------------
# Test Case 3: Chart Image Analysis
# ------------------------------------------------------------------
print("\n📌 TEST CASE 3 – Chart Image Analysis")
sample_image = "/path/to/your/chart.png"  # <-- CHANGE THIS

# Uncomment when you have a real image:
# resp3, tid3 = run_agent(f"Analyze the trends in this chart: {sample_image}", thread_id=tid1)

print("(Skipped – provide a real image path and uncomment the call)")

# ------------------------------------------------------------------
# Interactive multi-turn example (optional)
# ------------------------------------------------------------------
print("\n✅ Ready for interactive use.")
print("Example:")
print('  response, thread = run_agent("Summarize /content/report.pdf")')
print('  response, thread = run_agent("Now analyze this chart: /content/sales.png", thread_id=thread)')